# MAP Category+Misconception Analysis

This notebook analyzes the MAP dataset to create comprehensive definitions for each category+misconception label combination.

## Environment Setup

Setting up the working environment and configuring paths for the MAP dataset analysis.

In [ ]:
import os
import sys

# Configure data directory
DATA_DIR = "/kaggle/input/map-charting-student-math-misunderstandings"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "./data"

print(f"Data directory: {DATA_DIR}")
print(f"Python version: {sys.version}")

# Check if data directory exists
if os.path.exists(DATA_DIR):
    print(f"Files in data directory: {os.listdir(DATA_DIR)}")
else:
    print("Warning: Data directory not found!")

## Import Libraries

Importing all necessary libraries for data analysis and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from collections import Counter
from typing import Dict, List, Any

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('default')
sns.set_palette("husl")

## Configuration and Constants

Define parent category definitions and configuration variables.

In [ ]:
# Parent category definitions
CATEGORY_META_PROMPT = {
    "True_Correct": (
        "The student's response is correct (i.e. they do not have a misconception). "
        "It's 'True' meaning the ground-truth answer is valid. No conceptual error."
    ),
    "True_Neither": (
        "The student's response is not incorrect in a way that maps to a known misconception, "
        "but also not exactly 'fully correct'. This may capture ambiguous or edge cases "
        "where the explanation doesn't reveal correctness or a misconception clearly."
    ),
    "True_Misconception": (
        "The student's answer is technically correct, but their explanation reveals a misconception. "
        "They may have arrived at the right answer by flawed reasoning or a partial misunderstanding."
    ),
    "False_Misconception": (
        "The student's response is incorrect and the explanation indicates a specific misconception. "
        "This represents a clear case of misunderstanding tied to a known misconception category."
    ),
    "False_Neither": (
        "The student's response is incorrect, but the explanation does not map to a known misconception. "
        "Their reasoning is wrong, but not in a way that matches the predefined misconception labels."
    ),
    "False_Correct": (
        "The student's response is incorrect, but their explanation shows correct reasoning. "
        "This likely reflects a minor slip or arithmetic mistake rather than a conceptual error."
    )
}

# Column definitions
X_COL_LABELS = ['row_id', 'QuestionText', 'MC_Answer', 'StudentExplanation']
Y_COL_LABELS = ['Category:Misconception', 'y_correct_response', 'y_misconception_exist', 'y_misconception_type']
COL_LABELS = X_COL_LABELS + Y_COL_LABELS

print("Configuration loaded successfully!")
print(f"Parent categories: {list(CATEGORY_META_PROMPT.keys())}")

## Data Loading and Preprocessing

Load the training dataset and prepare it for analysis.

In [ ]:
def load_train_dataset(path: str = DATA_DIR + "/train.csv") -> pd.DataFrame:
    """Load and preprocess the training dataset."""

    print(f"Loading data from: {path}")
    train_data = pd.read_csv(path)
    print(f"Original dataset shape: {train_data.shape}")

    # Fill missing values
    train_data.fillna('NA', inplace=True)

    # Create combined category:misconception column
    train_data['Category:Misconception'] = train_data['Category'] + ":" + train_data['Misconception']

    # Extract components for analysis
    train_data['y_correct_response'] = train_data['Category'].str.split('_').str[0]
    train_data['y_misconception_exist'] = train_data['Category'].str.split('_').str[-1]
    train_data['y_misconception_type'] = train_data['Misconception']

    print(f"Processed dataset shape: {train_data.shape}")
    print(f"Columns: {list(train_data.columns)}")

    return train_data

# Load the dataset
train_data = load_train_dataset()

# Display basic info
print(f"\nUnique categories: {train_data['Category'].nunique()}")
print(f"Unique misconceptions: {train_data['Misconception'].nunique()}")
print(f"Unique category:misconception pairs: {train_data['Category:Misconception'].nunique()}")

train_data.head()

## Data Analysis and Exploration

Analyze the distribution of categories and misconceptions in the dataset.

In [ ]:
# Get unique combinations and their counts
combo_analysis = train_data.groupby(['Category', 'Misconception']).agg({
    'row_id': 'count',
    'StudentExplanation': lambda x: x.iloc[0]  # Get one example
}).rename(columns={'row_id': 'count', 'StudentExplanation': 'example_explanation'})

combo_analysis = combo_analysis.reset_index()
combo_analysis = combo_analysis.sort_values('count', ascending=False)

print("Top 20 most common Category:Misconception combinations:")
print(combo_analysis.head(20)[['Category', 'Misconception', 'count']])

# Get all unique category:misconception labels
all_labels = train_data['Category:Misconception'].unique().tolist()
print(f"\nTotal unique labels: {len(all_labels)}")

# Display all labels
print("\nAll Category:Misconception labels:")
for i, label in enumerate(sorted(all_labels), 1):
    count = train_data[train_data['Category:Misconception'] == label].shape[0]
    print(f"{i:2d}. {label:<40} (count: {count})")

In [ ]:
# Analyze misconception types by category
misconception_by_category = {}
for category in train_data['Category'].unique():
    misconceptions = train_data[train_data['Category'] == category]['Misconception'].unique()
    misconception_by_category[category] = list(misconceptions)
    print(f"\n{category} has {len(misconceptions)} unique misconceptions:")
    for misc in sorted(misconceptions):
        count = train_data[(train_data['Category'] == category) &
                          (train_data['Misconception'] == misc)].shape[0]
        print(f"  - {misc} (count: {count})")

## Misconception Definitions Generator

Create comprehensive definitions for each category+misconception combination.

In [ ]:
# Define specific misconception descriptions based on analysis of the data
MISCONCEPTION_DESCRIPTIONS = {
    "NA": "No specific misconception identified or not applicable",
    "Incomplete": "Student shows partial understanding but reasoning is incomplete or lacks key steps",
    "WNB": "When numerator bigger - incorrectly assumes operations when numerator is larger",
    "SwapDividend": "Swaps dividend and divisor in division operations",
    "Mult": "Uses multiplication instead of the correct operation",
    "FlipChange": "Incorrectly flips or inverts numbers/fractions during operations",
    "Irrelevant": "Focuses on irrelevant information or uses irrelevant reasoning",
    "Wrong_Fraction": "Misunderstands fraction concepts or representations",
    "Additive": "Uses additive reasoning where multiplicative reasoning is needed",
    "Not_variable": "Treats variables as fixed values rather than unknowns",
    "Adding_terms": "Incorrectly adds terms that cannot be combined",
    "Inverse_operation": "Uses the inverse of the required operation",
    "Inversion": "Inverts relationships or operations incorrectly",
    "Duplication": "Duplicates steps or counts elements multiple times",
    "Wrong_Operation": "Uses completely wrong mathematical operation",
    "Whole_numbers_larger": "Believes whole numbers are always larger than fractions/decimals",
    "Longer_is_bigger": "Assumes numbers with more digits are larger",
    "Ignores_zeroes": "Ignores trailing zeros or their significance",
    "Shorter_is_bigger": "Assumes numbers with fewer digits are larger",
    "Wrong_fraction": "Fundamental misunderstanding of fraction concepts",
    "Adding_across": "Adds numerators and denominators separately (cross addition)",
    "Denominator-only_change": "Only changes denominator when finding equivalent fractions",
    "Incorrect_equivalent_fraction_addition": "Misunderstands equivalent fractions in addition",
    "Division": "Uses division incorrectly or in wrong context",
    "Subtraction": "Uses subtraction incorrectly or in wrong context",
    "Unknowable": "Claims problem cannot be solved when it can be",
    "Definition": "Misunderstands basic mathematical definitions",
    "Interior": "Confuses interior and exterior elements or relationships",
    "Positive": "Assumes all results must be positive",
    "Tacking": "Adds extra elements or steps inappropriately",
    "Wrong_term": "Identifies or uses incorrect mathematical terms",
    "Firstterm": "Focuses only on first term while ignoring others",
    "Base_rate": "Misunderstands base rates or baseline comparisons",
    "Multiplying_by_4": "Incorrectly multiplies by 4 in specific contexts",
    "Certainty": "Treats probabilistic events as certain",
    "Scale": "Misunderstands scaling or proportional relationships"
}

def generate_category_misconception_definitions(train_data: pd.DataFrame) -> Dict[str, str]:
    """Generate comprehensive definitions for each category:misconception combination."""

    definitions = {}

    # Get all unique combinations
    unique_combos = train_data[['Category', 'Misconception']].drop_duplicates()

    for _, row in unique_combos.iterrows():
        category = row['Category']
        misconception = row['Misconception']
        combo_label = f"{category}:{misconception}"

        # Get base category description
        base_description = CATEGORY_META_PROMPT.get(category, "Unknown category")

        # Get misconception description
        misc_description = MISCONCEPTION_DESCRIPTIONS.get(misconception, f"Specific misconception: {misconception}")

        # Get sample count
        sample_count = train_data[
            (train_data['Category'] == category) &
            (train_data['Misconception'] == misconception)
        ].shape[0]

        # Create comprehensive definition
        if misconception == "NA":
            # For NA misconceptions, just use the base category description
            definition = f"{base_description}"
        else:
            # Combine base category with specific misconception
            definition = f"{base_description} Specifically, this involves: {misc_description.lower()}"

        definitions[combo_label] = {
            "definition": definition,
            "category": category,
            "misconception": misconception,
            "sample_count": sample_count,
            "category_description": base_description,
            "misconception_description": misc_description
        }

    return definitions

# Generate all definitions
all_definitions = generate_category_misconception_definitions(train_data)

print(f"Generated definitions for {len(all_definitions)} category:misconception combinations")
print("\nFirst 5 examples:")
for i, (label, info) in enumerate(list(all_definitions.items())[:5]):
    print(f"\n{i+1}. {label}")
    print(f"   Definition: {info['definition']}")
    print(f"   Sample count: {info['sample_count']}")

## Results Visualization

Create the final output dictionary and visualizations.

In [ ]:
# Create the final output dictionary with just the definitions
final_definitions = {label: info['definition'] for label, info in all_definitions.items()}

# Display as formatted JSON
print("=== FINAL CATEGORY:MISCONCEPTION DEFINITIONS ===")
print(json.dumps(final_definitions, indent=2, sort_keys=True))

# Save to file
output_file = "category_misconception_definitions.json"
with open(output_file, 'w') as f:
    json.dump(final_definitions, f, indent=2, sort_keys=True)

print(f"\n✅ Definitions saved to {output_file}")
print(f"📊 Total definitions: {len(final_definitions)}")

# Create summary statistics
categories = [info['category'] for info in all_definitions.values()]
misconceptions = [info['misconception'] for info in all_definitions.values()]

print(f"\n📈 Summary Statistics:")
print(f"   • Unique categories: {len(set(categories))}")
print(f"   • Unique misconceptions: {len(set(misconceptions))}")
print(f"   • Total combinations: {len(all_definitions)}")

# Show category distribution
print(f"\n📊 Category Distribution:")
category_counts = Counter(categories)
for cat, count in category_counts.most_common():
    print(f"   • {cat}: {count} misconception types")

In [ ]:
# Create visualization of the data distribution
plt.figure(figsize=(15, 10))

# Plot 1: Category distribution
plt.subplot(2, 2, 1)
category_counts = Counter([info['category'] for info in all_definitions.values()])
plt.bar(category_counts.keys(), category_counts.values())
plt.title('Number of Misconception Types per Category')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Number of Misconception Types')

# Plot 2: Sample count distribution
plt.subplot(2, 2, 2)
sample_counts = [info['sample_count'] for info in all_definitions.values()]
plt.hist(sample_counts, bins=20, edgecolor='black')
plt.title('Distribution of Sample Counts')
plt.xlabel('Sample Count')
plt.ylabel('Frequency')

# Plot 3: Top misconceptions
plt.subplot(2, 2, 3)
misc_counts = Counter([info['misconception'] for info in all_definitions.values()])
top_misc = dict(misc_counts.most_common(10))
plt.barh(list(top_misc.keys()), list(top_misc.values()))
plt.title('Top 10 Most Common Misconceptions')
plt.xlabel('Number of Categories')

# Plot 4: Sample counts by category
plt.subplot(2, 2, 4)
category_sample_counts = {}
for info in all_definitions.values():
    cat = info['category']
    if cat not in category_sample_counts:
        category_sample_counts[cat] = 0
    category_sample_counts[cat] += info['sample_count']

plt.bar(category_sample_counts.keys(), category_sample_counts.values())
plt.title('Total Sample Counts by Category')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Total Sample Count')

plt.tight_layout()
plt.savefig('category_analysis_plots.png', dpi=300, bbox_inches='tight')
plt.show()

print("📊 Visualizations created and saved as 'category_analysis_plots.png'")